<a href="https://colab.research.google.com/github/ricardoserodio/portugal-term-deposit-comparator/blob/main/notebooks/portugal_term_deposit_comparator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import files
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import ipywidgets as widgets

uploaded = files.upload()

Saving depositos_prazo_core_portugal_corrigido_validado.xlsx to depositos_prazo_core_portugal_corrigido_validado (1).xlsx


In [ ]:
file_name = "depositos_prazo_core_portugal_corrigido_validado.xlsx"

df = pd.read_excel(file_name, sheet_name="Tabela_Core")

df.head()

In [24]:
df_calc = df.copy()

# Garantir colunas numéricas
df_calc["TANB (%)"] = pd.to_numeric(df_calc["TANB (%)"], errors="coerce")
df_calc["Prazo (meses)"] = pd.to_numeric(df_calc["Prazo (meses)"], errors="coerce")
df_calc["Mínimo (€)"] = pd.to_numeric(df_calc["Mínimo (€)"], errors="coerce")
df_calc["Máximo (€)"] = pd.to_numeric(df_calc["Máximo (€)"], errors="coerce")
df_calc["Taxa IRS"] = pd.to_numeric(df_calc["Taxa IRS"], errors="coerce")

# Limpar colunas de texto
text_cols = [
    "Banco",
    "Produto",
    "Só novos clientes",
    "Só novos montantes",
    "Mobilização antecipada",
    "IRS aplicável",
    "Notas / condições",
    "Validação rápida",
    "Observação de validação",
    "Fonte oficial / referência",
]

for col in text_cols:
    if col in df_calc.columns:
        df_calc[col] = df_calc[col].fillna("").astype(str).str.strip()

df_calc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Banco                       43 non-null     object 
 1   Produto                     43 non-null     object 
 2   Prazo (meses)               43 non-null     int64  
 3   TANB (%)                    43 non-null     float64
 4   Mínimo (€)                  39 non-null     float64
 5   Máximo (€)                  31 non-null     float64
 6   Só novos clientes           43 non-null     object 
 7   Só novos montantes          43 non-null     object 
 8   Mobilização antecipada      43 non-null     object 
 9   IRS aplicável               43 non-null     object 
 10  Taxa IRS                    43 non-null     float64
 11  Notas / condições           43 non-null     object 
 12  Validação rápida            43 non-null     object 
 13  Observação de validação     43 non-nu

In [25]:
def gerar_alertas(row):
    alertas = []

    if str(row["Só novos clientes"]).lower() in ["sim", "ver notas"]:
        alertas.append("Só novos clientes")

    if str(row["Só novos montantes"]).lower() == "sim":
        alertas.append("Só novos montantes")

    if str(row["Mobilização antecipada"]).lower() == "ver notas":
        alertas.append("Mobilização antecipada: ver notas")

    if pd.notna(row["Máximo (€)"]):
        alertas.append("Tem montante máximo")

    if str(row["Validação rápida"]).lower() != "validado rápido":
        alertas.append("Validação parcial")

    if len(alertas) == 0:
        return "Sem alertas relevantes"

    return " | ".join(alertas)


def comparar_depositos(
    capital,
    prazo_meses=None,
    exigir_mobilizacao=False,
    aceitar_so_novos_clientes=True,
    aceitar_so_novos_montantes=True,
    top_n=10
):
    resultado = df_calc.copy()

    # Filtrar por capital mínimo
    resultado = resultado[
        resultado["Mínimo (€)"].isna() | (capital >= resultado["Mínimo (€)"])
    ]

    # Filtrar por capital máximo
    resultado = resultado[
        resultado["Máximo (€)"].isna() | (capital <= resultado["Máximo (€)"])
    ]

    # Filtrar por prazo
    if prazo_meses is not None:
        resultado = resultado[resultado["Prazo (meses)"] == prazo_meses]

    # Exigir mobilização antecipada
    if exigir_mobilizacao:
        resultado = resultado[
            resultado["Mobilização antecipada"].str.lower().isin(["sim", "ver notas"])
        ]

    # Excluir produtos só para novos clientes
    if not aceitar_so_novos_clientes:
        resultado = resultado[
            ~resultado["Só novos clientes"].str.lower().isin(["sim", "ver notas"])
        ]

    # Excluir produtos só para novos montantes
    if not aceitar_so_novos_montantes:
        resultado = resultado[
            resultado["Só novos montantes"].str.lower() != "sim"
        ]

    # Cálculos financeiros
    resultado["Juro bruto estimado (€)"] = (
        capital
        * (resultado["TANB (%)"] / 100)
        * (resultado["Prazo (meses)"] / 12)
    )

    resultado["IRS estimado (€)"] = (
        resultado["Juro bruto estimado (€)"] * resultado["Taxa IRS"]
    )

    resultado["Juro líquido estimado (€)"] = (
        resultado["Juro bruto estimado (€)"] - resultado["IRS estimado (€)"]
    )

    resultado["Montante final estimado (€)"] = (
        capital + resultado["Juro líquido estimado (€)"]
    )

    # Alertas
    resultado["Alertas"] = resultado.apply(gerar_alertas, axis=1)

    # Arredondar valores
    money_cols = [
        "Juro bruto estimado (€)",
        "IRS estimado (€)",
        "Juro líquido estimado (€)",
        "Montante final estimado (€)"
    ]

    resultado[money_cols] = resultado[money_cols].round(2)

    # Ordenar por melhor juro líquido
    resultado = resultado.sort_values(
        by="Juro líquido estimado (€)",
        ascending=False
    )

    colunas_finais = [
        "Banco",
        "Produto",
        "Prazo (meses)",
        "TANB (%)",
        "Mínimo (€)",
        "Máximo (€)",
        "Só novos clientes",
        "Só novos montantes",
        "Mobilização antecipada",
        "Juro bruto estimado (€)",
        "IRS estimado (€)",
        "Juro líquido estimado (€)",
        "Montante final estimado (€)",
        "Alertas",
        "Notas / condições",
        "Validação rápida",
        "Fonte oficial / referência"
    ]

    return resultado[colunas_finais].head(top_n)


def ranking_resumido(resultado):
    colunas_resumo = [
        "Banco",
        "Produto",
        "Prazo (meses)",
        "TANB (%)",
        "Juro líquido estimado (€)",
        "Montante final estimado (€)",
        "Alertas",
    ]

    return resultado[colunas_resumo]

In [26]:
resultado_teste = comparar_depositos(
    capital=10000,
    prazo_meses=12,
    exigir_mobilizacao=False,
    aceitar_so_novos_clientes=True,
    aceitar_so_novos_montantes=True,
    top_n=10
)

ranking_resumido(resultado_teste)

,Banco,Produto,Prazo (meses),TANB (%),Juro líquido estimado (€),Montante final estimado (€),Alertas
20,Haitong Bank,Depósito a Prazo Haitong,12,2.40,172.8,10172.8,Mobilização antecipada: ver notas
35,Banco BNI Europa,Depósito a Prazo,12,2.20,158.4,10158.4,Mobilização antecipada: ver notas
37,Banco BAI Europa,Depósito a Prazo Premium EUR,12,2.15,154.8,10154.8,Tem montante máximo
38,Banco Português de Gestão (BPG),BPG Valor,12,2.15,154.8,10154.8,Tem montante máximo


In [27]:
capital_widget = widgets.FloatText(
    value=10000,
    description="Capital (€):",
    style={"description_width": "initial"}
)

prazo_widget = widgets.Dropdown(
    options=sorted(df_calc["Prazo (meses)"].dropna().unique()),
    value=12,
    description="Prazo:",
    style={"description_width": "initial"}
)

mobilizacao_widget = widgets.Checkbox(
    value=False,
    description="Exigir mobilização antecipada"
)

novos_clientes_widget = widgets.Checkbox(
    value=True,
    description="Aceitar produtos só para novos clientes"
)

novos_montantes_widget = widgets.Checkbox(
    value=True,
    description="Aceitar produtos só para novos montantes"
)

top_widget = widgets.IntSlider(
    value=10,
    min=3,
    max=20,
    step=1,
    description="Top N:",
    style={"description_width": "initial"}
)

botao = widgets.Button(
    description="Comparar depósitos",
    button_style="success"
)

output = widgets.Output()


def on_button_clicked(b):
    with output:
        output.clear_output()

        resultado = comparar_depositos(
            capital=capital_widget.value,
            prazo_meses=prazo_widget.value,
            exigir_mobilizacao=mobilizacao_widget.value,
            aceitar_so_novos_clientes=novos_clientes_widget.value,
            aceitar_so_novos_montantes=novos_montantes_widget.value,
            top_n=top_widget.value
        )

        display(Markdown("## Ranking de depósitos a prazo"))
        display(ranking_resumido(resultado))


botao.on_click(on_button_clicked)

display(
    capital_widget,
    prazo_widget,
    mobilizacao_widget,
    novos_clientes_widget,
    novos_montantes_widget,
    top_widget,
    botao,
    output
)

FloatText(value=10000.0, description='Capital (€):', style=DescriptionStyle(description_width='initial'))

Dropdown(description='Prazo:', index=3, options=(np.int64(3), np.int64(6), np.int64(9), np.int64(12), np.int64…

Checkbox(value=False, description='Exigir mobilização antecipada')

Checkbox(value=True, description='Aceitar produtos só para novos clientes')

Checkbox(value=True, description='Aceitar produtos só para novos montantes')

IntSlider(value=10, description='Top N:', max=20, min=3, style=SliderStyle(description_width='initial'))

Button(button_style='success', description='Comparar depósitos', style=ButtonStyle())

Output()

In [29]:
df_calc.to_csv("depositos_prazo_core_portugal_corrigido.csv", index=False, encoding="utf-8-sig")

In [31]:
from google.colab import files

files.download("depositos_prazo_core_portugal_corrigido.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>